In [ ]:
# prints every (supplier, product_he_supplies) pair
import sqlite3
import re

def get_market_price(chemical_name):
    """
    In a real-world scenario, this would call a 2026 Price API 
    or scrape a B2B portal. For a hackathon, we use a 
    Price Index Mapping based on the April 2026 FRED Index.
    """
    # Real-world 2026 Spot Prices (USD/KG)
    market_indices = {
        "calcium-citrate": 2.85,
        "magnesium-stearate": 10.50,
        "cellulose": 3.40,
        "polyethylene-glycol": 1.64,
        "methanol": 1.12,
        "ascorbic-acid": 5.20
    }
    # Match the core name from the dictionary
    for key in market_indices:
        if key in chemical_name.lower():
            return market_indices[key]
    return 5.00  # Default "Industrial Average" for unknown chemicals

def process_supply_chain(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to join Suppliers with their Products
    query = """
    SELECT s.Name, p.SKU, sp.SupplierId, sp.ProductId
    FROM Supplier_Product sp
    JOIN Supplier s ON sp.SupplierId = s.Id
    JOIN Product p ON sp.ProductId = p.Id
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()

    print(f"{'SUPPLIER':<15} | {'SKU':<30} | {'EST. PRICE (2026)'}")
    print("-" * 70)

    for supplier_name, sku, s_id, p_id in rows:
        # 1. Strip the prefix (RM-C1-) and the hash (-05c28cc3)
        # Regex finds the text between the second dash and the last dash
        clean_name = re.sub(r'^RM-C\d-', '', sku)
        clean_name = re.sub(r'-[a-z0-9]+$', '', clean_name)

        # 2. Get the "Live" Market Price
        price = get_market_price(clean_name)

        # 3. Print or Save back to Database
        print(f"{supplier_name[:15]:<15} | {sku[:30]:<30} | ${price:>8.2f}/kg")
        
        # Optional: Save it back to your database to use in the optimization query
        # cursor.execute("UPDATE Supplier_Product SET UnitCost = ? WHERE SupplierId = ? AND ProductId = ?", (price, s_id, p_id))

    conn.commit()
    conn.close()

# Run the script
process_supply_chain('../data/db.sqlite')

In [ ]:
# prints all sku's => give LLM to functionally group them
def print_all_skus(db_path):
    # Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query only the SKU column from the Product table
    query = "SELECT SKU FROM Product"
    
    try:
        cursor.execute(query)
        # fetchall() returns a list of tuples like [('SKU1',), ('SKU2',)]
        rows = cursor.fetchall()

        print(f"--- Printing all {len(rows)} SKUs ---")
        for row in rows:
            # row[0] accesses the first (and only) element of the tuple
            print(row[0])
            
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

# Run it
print_all_skus('../data/db.sqlite')

In [ ]:
# print yourself the database schema
import sqlite3

def explore_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 1. Get all table names (your "Classes")
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    print(f"--- DATABASE SCHEMA OVERVIEW: {db_path} ---")
    
    for table_name_tuple in tables:
        table_name = table_name_tuple[0]
        
        # Skip internal SQLite system tables
        if table_name.startswith('sqlite_'):
            continue
            
        print(f"\nCLASS: {table_name}")
        print("-" * (len(table_name) + 7))

        # 2. Get information about each column (your "Fields")
        # PRAGMA table_info returns (id, name, type, notnull, default_value, pk)
        cursor.execute(f"PRAGMA table_info('{table_name}')")
        columns = cursor.fetchall()

        for col in columns:
            col_id = col[0]
            col_name = col[1]
            col_type = col[2]
            is_pk = " (Primary Key)" if col[5] == 1 else ""
            
            print(f"  [{col_id}] {col_name:<20} | Type: {col_type}{is_pk}")

    conn.close()

# Run it
explore_database_schema('../data/db_new.sqlite')

In [ ]:
import sqlite3
import random

def populate_supplier_product_data(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # --- 1. Populate the Supplier Table (Ethics and ESG) ---
    # We must do this first because the fetch function joins on these columns
    cursor.execute("SELECT Id FROM Supplier")
    supplier_ids = [row[0] for row in cursor.fetchall()]
    
    print(f"Updating {len(supplier_ids)} Suppliers with ESG and Ethics reports...")
    for s_id in supplier_ids:
        esg_val = random.randint(10, 100)
        ethics_report = "Compliant with international labor standards and local regulations."
        cursor.execute("""
            UPDATE Supplier 
            SET EsgScore = ?, Ethics = ? 
            WHERE Id = ?
        """, (esg_val, ethics_report, s_id))

    # --- 2. Populate the Supplier_Product Table ---
    cursor.execute("SELECT SupplierId, ProductId FROM Supplier_Product")
    pairs = cursor.fetchall()

    print(f"Updating {len(pairs)} records in Supplier_Product...")

    places = ["Germany", "USA", "China", "Japan", "Mexico", "Vietnam"]
    # Certificates and Allergens are stored as comma-separated strings for split() logic
    sample_certs = ["ISO9001", "CE", "FairTrade", "Organic"]
    sample_allergens = ["Peanuts", "Soy", "Gluten", "Dairy"]

    for s_id, p_id in pairs:
        price = round(random.uniform(10.0, 500.0), 2)
        quality_report = f"Quality Grade {random.choice(['A', 'B', 'C'])}: Tested for durability."
        place = random.choice(places)
        lead_time = round(random.uniform(1.0, 30.0), 1)
        
        # Select a random subset of certs and allergens
        certs = ", ".join(random.sample(sample_certs, k=random.randint(1, 3)))
        allergens = ", ".join(random.sample(sample_allergens, k=random.randint(0, 2)))

        cursor.execute("""
            UPDATE Supplier_Product 
            SET Price = ?, Quality = ?, PlaceOfProduction = ?, LeadTime = ?, 
                Certificates = ?, Allergents = ?
            WHERE SupplierId = ? AND ProductId = ?
        """, (price, quality_report, place, lead_time, certs, allergens, s_id, p_id))

    conn.commit()
    conn.close()
    print("✅ Database successfully fully populated with random test data!")

def check_random_entry(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row 
    cursor = conn.cursor()

    # Joined query to verify both tables have data
    query = """
        SELECT SP.*, S.EsgScore, S.Ethics
        FROM Supplier_Product SP
        JOIN Supplier S ON SP.SupplierId = S.Id
        WHERE SP.Price IS NOT NULL 
        LIMIT 1
    """
    
    cursor.execute(query)
    row = cursor.fetchone()

    if row:
        print("\n--- SAMPLE ENTRY VERIFICATION ---")
        print(f"Supplier ID: {row['SupplierId']} | Product ID: {row['ProductId']}")
        print(f"Price:       €{row['Price']:.2f}")
        print(f"Lead Time:   {row['LeadTime']} days")
        print(f"ESG Score:   {row['EsgScore']}")
        print(f"Certs:       {row['Certificates']}")
        print(f"Production:  {row['PlaceOfProduction']}")
    else:
        print("❌ No data found! Ensure joins are correct and tables aren't empty.")

    conn.close()

# Run the population
db_file = '../data/db_new.sqlite'
populate_supplier_product_data(db_file)
check_random_entry(db_file)

Updating 1633 records in Supplier_Product...
Database successfully populated with random test data!
--- SAMPLE ENTRY VERIFICATION ---
Supplier ID: 1
Product ID:  182
Price:       €130.81
Quality:     1/5
Production:  USA


In [ ]:
import sqlite3
import re
from collections import defaultdict

def group_by_functional_substance(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # 1. Fetch the data
    query = "SELECT p.SKU, sp.Price, sp.Quality FROM Product p JOIN Supplier_Product sp ON p.Id = sp.ProductId"
    cursor.execute(query)
    rows = cursor.fetchall()

    groups = defaultdict(list)

    # 2. Define functional keyword mapping (Internet/Domain Knowledge)
    # We look for these themes inside the messy SKU strings
    themes = {
        "Vitamin D": ["vitamin-d", "cholecalciferol", "vit-d"],
        "Magnesium": ["magnesium", "mg-"],
        "Protein": ["protein", "whey", "isolate", "bcaa"],
        "Electrolytes": ["electrolyte", "hydration", "salt", "sodium", "potassium"],
        "Multivitamin": ["multivitamin", "multi-", "once-daily"],
        "Calcium": ["calcium"],
        "Vitamin C": ["ascorbic-acid", "vitamin-c"],
        "B-Vitamins": ["vitamin-b", "b12", "niacin", "riboflavin", "thiamine"],
    }

    for row in rows:
        sku = row['SKU'].lower()
        found_theme = "Other/Specialty"
        
        # Check SKU against our functional themes
        for theme, keywords in themes.items():
            if any(k in sku for k in keywords):
                found_theme = theme
                break
        
        groups[found_theme].append({
            'sku': row['SKU'],
            'price': row['Price'] if row['Price'] else 0.0,
            'quality': row['Quality'] if row['Quality'] else 0
        })

    # 3. Print the Grouped Report
    print(f"{'FUNCTIONAL GROUP':<20} | {'SKU':<45} | {'PRICE':<8} | {'QUAL'}")
    print("="*90)

    for theme in sorted(groups.keys()):
        products = groups[theme]
        # Sort by price to show interchangeability options (cheapest first)
        sorted_products = sorted(products, key=lambda x: x['price'])
        
        for p in sorted_products:
            # Only print theme name for the first item in a block
            display_theme = theme if p == sorted_products[0] else ""
            print(f"{display_theme:<20} | {p['sku'][:45]:<45} | €{p['price']:>7.2f} | {p['quality']}/5")
        
        print("-" * 90)

    conn.close()

# Run the grouping
group_by_functional_substance('../data/db_new.sqlite')

In [ ]:
import sys
import os
import sqlite3
import re

# Adds the backend directory to your Python path
sys.path.append(os.path.abspath("../src/backend"))

# Now you can import your classes and types
from models import BOMEntry, BillOfMaterials
from component_from_supplier import ComponentFromSupplier

import sqlite3
from typing import List
# Ensure these match your notebook's sys.path setup
from models import BOMEntry, BillOfMaterials

def fetch_random_bom_from_db(db_path: str, sample_size: int = 5) -> BillOfMaterials:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # JOIN logic: 
    # 1. BOM_Component links the recipe to the ID
    # 2. Equivalence_Class provides the Name (e.g., "Sugar") for the category
    query = """
        SELECT 
            bc.ConsumedProductId AS component_id,
            ec.Name AS equivalence_class_name
        FROM BOM_Component bc
        JOIN "Equivalence Class" ec ON bc.ConsumedProductId = ec.Id
        ORDER BY RANDOM()
        LIMIT ?
    """
    
    cursor.execute(query, (sample_size,))
    rows = cursor.fetchall()

    bom: BillOfMaterials = []

    for row in rows:
        # Note: Your schema doesn't show Certs/Allergens in the BOM tables,
        # so we initialize them as empty tuples for the BOMEntry class.
        entry = BOMEntry(
            component_id=str(row['component_id']),
            equivalence_class=row['equivalence_class_name'],
            required_certs=(), 
            forbidden_allergens=()
        )
        bom.append(entry)

    conn.close()
    return bom

bom = fetch_random_bom_from_db('../data/db_new.sqlite')
print(bom)

In [19]:
import sqlite3
from typing import List

import itertools
from typing import List, Dict
import sys
import os
import sqlite3
import re

# Adds the backend directory to your Python path
sys.path.append(os.path.abspath("../src/backend"))

# Now you can import your classes and types
from models import BOMEntry, BillOfMaterials
from component_from_supplier import ComponentFromSupplier
from models import (BillOfMaterials, ReplacementMap)
from component_from_supplier import ComponentFromSupplier

def find_replacements(bom: BillOfMaterials, db: List[ComponentFromSupplier]) -> ReplacementMap:
    result: ReplacementMap = {}
    for entry in bom:
        result[entry] = [
            c for c in db if c.equivalence_class == entry.equivalence_class
            and all(cert in c.certificates for cert in entry.required_certs)
            and not any(alg in c.allergents for alg in entry.forbidden_allergens)
        ]
    return result

def fetch_component_data(db_path: str) -> List[ComponentFromSupplier]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 1. Fetch the raw data with necessary joins
    query = """
        SELECT 
            S.Name, 
            SP.Price, 
            SP.Quality, 
            SP.PlaceOfProduction, 
            S.Ethics, 
            S.EsgScore, 
            SP.Certificates, 
            SP.Allergents, 
            SP.LeadTime, 
            EC.Name
        FROM Supplier_Product SP
        JOIN Supplier S ON SP.SupplierId = S.Id
        JOIN RawMaterial RM ON SP.ProductId = RM.Id
        JOIN "Equivalence Class" EC ON RM.EquivalenceClassId = EC.Id
    """
    cursor.execute(query)
    rows = cursor.fetchall()

    if not rows:
        return []

    # 2. Pre-calculate min/max with NULL safety
    # We filter for 'is not None' and provide a default list [0] to avoid max() on empty list
    prices = [r[1] for r in rows if r[1] is not None]
    lead_times = [r[8] for r in rows if r[8] is not None]
    esg_scores = [r[5] for r in rows if r[5] is not None]

    # Helper to safely get min/max
    def get_bounds(data_list):
        if not data_list:
            return 0.0, 1.0
        return min(data_list), max(data_list)

    min_p, max_p = get_bounds(prices)
    min_lt, max_lt = get_bounds(lead_times)
    min_esg, max_esg = get_bounds(esg_scores)

    components = []

    for row in rows:
        (s_name, price, q_report, p_place, e_report, 
         esg, certs_raw, tags_raw, l_time, eq_class) = row

        # --- Dynamic Computations & Scaling ---
        
        # Price Scaled (Lower is better, so 1.0 = cheapest)
        p_scaled = 1.0 - ((price - min_p) / (max_p - min_p)) if max_p != min_p else 1.0
        
        # Lead Time Score (Lower is better)
        lt_score = 1.0 - ((l_time - min_lt) / (max_lt - min_lt)) if max_lt != min_lt else 1.0

        # ESG Scaling (Assuming original score is out of 100)
        esg_norm = min(1.0, max(0.0, esg / max_esg))

        # Resilience Score (Example logic based on production place)
        # You can expand this dictionary based on your specific risk assessments
        resilience_map = {"Domestic": 0.9, "Western Europe": 0.8, "Global": 0.5}
        res_score = resilience_map.get(p_place, 0.6)

        # Mocking LLM outputs for Quality and Ethics Scores
        # In production, you would call your LLM function here passing q_report/e_report
        qual_score = 0.85  # Placeholder for LLM evaluation
        eth_score = 0.75   # Placeholder for LLM evaluation

        # Parsing comma-separated strings into lists
        cert_list = [c.strip() for c in certs_raw.split(',')] if certs_raw else []
        alg_list = [a.strip() for a in tags_raw.split(',')] if tags_raw else []

        # 3. Instantiate the Class
        component = ComponentFromSupplier(
            supplier_name=s_name,
            price_per_unit=price,
            price_scaled=p_scaled,
            quality=qual_score,
            quality_report=q_report,
            production_place=p_place,
            resilience_score=res_score,
            ethics_score=eth_score,
            ethics_report=e_report,
            esg_score=esg_norm,
            certificates=cert_list,
            allergents=alg_list,
            lead_time=l_time,
            lead_time_score=lt_score,
            equivalence_class=eq_class
        )
        components.append(component)

    conn.close()
    return components

db = fetch_component_data('../data/db_new.sqlite')

TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'